# 99 - Reset demo to baseline

**Destructive for the demo tables.** Use before a rehearsal or a live session.

1. Removes every landing file added by `01_simulate_increments` (baseline files from `00_setup_landing` are kept).
2. Optionally runs a **full refresh** of the sdp-meta pipelines (bronze for both groups, then silver), then a regular update of the gold pipeline.
   A full refresh truncates the streaming tables and resets their Auto Loader checkpoints, so tables return to baseline counts.

If you changed `conf/` (for example re-commented `suppliers`), run the onboarding job **before** the full refresh so the dataflowspec tables match.

In [0]:
dbutils.widgets.text("catalog", "mg_aws_serverless_workspace_catalog", "Catalog")
dbutils.widgets.text("prefix", "sdp_meta", "Schema prefix")
dbutils.widgets.text("sales_stream_baseline_files", "15", "sales_stream files in baseline")
dbutils.widgets.dropdown("target", "dev", ["dev", "prod"], "Bundle target")
dbutils.widgets.dropdown("full_refresh", "yes", ["yes", "no"], "Full refresh pipelines")

CATALOG = dbutils.widgets.get("catalog")
PREFIX = dbutils.widgets.get("prefix")
STREAM_BASELINE = int(dbutils.widgets.get("sales_stream_baseline_files"))
TARGET = dbutils.widgets.get("target")
FULL_REFRESH = dbutils.widgets.get("full_refresh") == "yes"

SRC = "/Volumes/samples/databricks/datasets/retail-org"
LANDING = f"/Volumes/{CATALOG}/{PREFIX}_landing/files"

# Bundle `mode: development` prefixes resource names with "[dev <user_short_name>] ".
user = spark.sql("SELECT current_user()").first()[0]
NAME_PREFIX = f"[dev {user.split('@')[0].replace('.', '_')}] " if TARGET == "dev" else ""
print(LANDING, repr(NAME_PREFIX), sep="\n")

In [0]:
stream_src = sorted(f.name for f in dbutils.fs.ls(f"{SRC}/sales_stream/sales_stream.json"))
KEEP = {
    "retail_reference/customers": {"customers_000_initial.csv"},
    "retail_sales/sales_orders": {"sales_orders_000_initial.json"},
    "retail_sales/sales_stream": set(stream_src[:STREAM_BASELINE]),
}
removed = []
for folder, keep in KEEP.items():
    for f in dbutils.fs.ls(f"{LANDING}/{folder}"):
        if f.name not in keep:
            dbutils.fs.rm(f.path)
            removed.append(f"{folder}/{f.name}")
print(f"removed {len(removed)} files", *removed[:20], sep="\n  ")

# Check: each folder now holds exactly the baseline files
for folder, keep in KEEP.items():
    left = {f.name for f in dbutils.fs.ls(f"{LANDING}/{folder}")}
    assert left == keep, f"{folder}: expected {len(keep)} baseline files, found {sorted(left - keep)} extra / {sorted(keep - left)} missing"
print("landing zone matches baseline")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()


def pipeline_id(name):
    """Look up a bundle-deployed pipeline id by its exact name for this target."""
    full = NAME_PREFIX + name
    hits = [p for p in w.pipelines.list_pipelines(filter=f"name LIKE '%{name}'") if p.name == full]
    assert len(hits) == 1, f"expected exactly one pipeline named {full!r}, found {len(hits)}"
    return hits[0].pipeline_id


def full_refresh(names, full=True):
    """Start (full-refresh) updates for the given pipelines in parallel and wait for all to finish."""
    started = {n: (pid := pipeline_id(n), w.pipelines.start_update(pid, full_refresh=full).update_id) for n in names}
    for n, (pid, uid) in started.items():
        while (state := w.pipelines.get_update(pid, uid).update.state.value) not in ("COMPLETED", "FAILED", "CANCELED"):
            time.sleep(15)
        print(f"{n}: {state}")
        assert state == "COMPLETED", f"{n} full refresh ended in {state}"


import time

if FULL_REFRESH:
    full_refresh(["sdp_meta_bronze_retail_reference", "sdp_meta_bronze_retail_sales"])
    full_refresh(["sdp_meta_silver_retail_reference", "sdp_meta_silver_retail_sales"])
    # Gold is materialized views only: a regular update recomputes them from the reset silver tables.
    full_refresh(["sdp_meta_gold_retail"], full=False)
else:
    print("skipped full refresh (full_refresh=no)")

In [0]:
tables = spark.sql(
    "SELECT table_schema, table_name FROM system.information_schema.tables "
    "WHERE table_catalog = :c AND table_schema IN (:b, :s, :g) ORDER BY 1, 2",
    args={"c": CATALOG, "b": f"{PREFIX}_bronze", "s": f"{PREFIX}_silver", "g": f"{PREFIX}_gold"},
).collect()
counts = [(t.table_schema, t.table_name, spark.table(f"`{CATALOG}`.`{t.table_schema}`.`{t.table_name}`").count()) for t in tables]
display(spark.createDataFrame(counts, "schema STRING, table STRING, rows LONG"))